# Projektaufgabe: <Titel>

**Vorlesung:** Innovative Konzepte zur Programmierung von Industrierobotern  
**Dozent:** Prof. Dr.-Ing. Björn Hein  
**Gruppe:** <Namen>  
**Aufgabe:** <Nummer und Titel>  
**Abgabedatum:** <Datum>  
**Vortragsdatum:** <Datum>

## 1. Kurzfassung

Fassen Sie kurz Problem, Ansatz, wichtigste Ergebnisse und offene Punkte zusammen.

## 2. Zielsetzung

Beschreiben Sie die konkrete Aufgabenstellung, eigene Teilziele und experimentelle Fragestellungen.

## 3. Ausgangscode und verwendete Module

Dokumentieren Sie verwendete Module, erweiterte Klassen/Funktionen, neue Module und den Startpunkt der eigenen Lösung.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
NOTEBOOKS_DIR = REPO_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.append(str(NOTEBOOKS_DIR))

REPO_ROOT

## 4. Konzept und Algorithmus

Beschreiben Sie den Algorithmus fachlich. Nutzen Sie Skizzen, Pseudocode, Tabellen oder kurze Formeln, wenn sie helfen.

## 5. Implementierung

Der Roundtrip-Algorithmus wird als separates Python-Modul eingebunden. Dieses Notebook definiert eine kleine Integrationsschnittstelle, damit Benchmarks, Visualisierung, Animation und Auswertung unabhängig von internen Implementierungsdetails bleiben. Nach dem Import muss lediglich eine Factory registriert werden, die eine Basisplaner-Klasse und einen CollisionChecker entgegennimmt.


In [ ]:
# TASK2_INTEGRATION_CONTRACT
from copy import deepcopy

from notebooks.IPBasicPRM import BasicPRM
from notebooks.IPLazyPRM import LazyPRM
from notebooks.IPVisibilityPRM import VisPRM


BASE_PLANNER_CLASSES = {
    "BasicPRM": BasicPRM,
    "LazyPRM": LazyPRM,
    "VisibilityPRM": VisPRM,
}

BASE_PLANNER_CONFIGS = {
    "BasicPRM": {
        "radius": 5.0,
        "numNodes": 300,
        "collisionCheckingSteps": 40,
        "useKDTree": True,
    },
    "LazyPRM": {
        "initialRoadmapSize": 40,
        "updateRoadmapSize": 20,
        "kNearest": 5,
        "maxIterations": 40,
    },
    "VisibilityPRM": {
        "ntry": 40,
    },
}

# Set this factory after Dima's implementation is available.
ROUNDTRIP_PLANNER_FACTORY = None


def configure_roundtrip_algorithm(factory):
    """Register a factory accepting a base-planner class and collision checker."""
    if not callable(factory):
        raise TypeError("The RoundtripPlanner factory must be callable.")

    global ROUNDTRIP_PLANNER_FACTORY
    ROUNDTRIP_PLANNER_FACTORY = factory


def create_roundtrip_planner(base_planner_name, benchmark):
    """Create a configured RoundtripPlanner through the integration boundary."""
    if ROUNDTRIP_PLANNER_FACTORY is None:
        raise RuntimeError(
            "No RoundtripPlanner is configured. Import Dima's implementation "
            "and call configure_roundtrip_algorithm(...) first."
        )

    base_planner_class = BASE_PLANNER_CLASSES[base_planner_name]
    return ROUNDTRIP_PLANNER_FACTORY(
        base_planner_class,
        benchmark.collisionChecker,
    )


def build_roundtrip_config(base_planner_name, order_method, seed):
    """Build one reproducible configuration for a RoundtripPlanner run."""
    return {
        "basePlannerConfig": deepcopy(
            BASE_PLANNER_CONFIGS[base_planner_name]
        ),
        "orderMethod": order_method,
        "randomSeed": seed,
        "pairRetries": 2,
    }


In [ ]:
import traceback

from IPVISVisibilityPRM import visibilityPRMVisualize
from src.multiquery.MultiqueryPRMVisualize import multiqueryPRMVisualize

from src.multiquery.VisibilityPRMRoadmapper import VisibilityPRMRoadmapper
from src.multiquery.MultiQueryRoundtripPlanner import MultiQueryRoundtripPlanner

import src.benchmarking.RoundtripTestSuite as ts

import matplotlib.pylab as plt


visConfig = dict()
visConfig["ntry"] = 50
visConfig["mConnections"] = True
visConfig["directConnections"] = True

for benchmark in ts.benchList:
    try:
        fig_local = plt.figure(figsize=(10,10))
        ax = fig_local.add_subplot(1,1,1)
        planner = MultiQueryRoundtripPlanner(VisibilityPRMRoadmapper(benchmark.collisionChecker), benchmark.collisionChecker)
        solution = planner.planPath(benchmark.startList, benchmark.goalList, visConfig)
        title = benchmark.name
        if solution == []:
            title += " (No path found!)"
        print(solution)
        title += "\n Assumed complexity level " + str(benchmark.level)
        ax.set_title(title)
        multiqueryPRMVisualize(planner, solution, ax=ax, nodeSize=50)
    except Exception as e:
        print("ERROR: ",benchmark.name, e)
        for frame in traceback.extract_tb(e.__traceback__):
            print(frame.filename, frame.lineno, frame.name)

## 6. Validierung an kleinen Beispielen

Zeigen Sie an kleinen, nachvollziehbaren Fällen, dass die zentralen Bausteine korrekt funktionieren.

In [ ]:
REQUIRED_ROUNDTRIP_ATTRIBUTES = (
    "graph",
    "plannerClass",
    "planners",
    "paths",
)


def validate_roundtrip_interface(planner, solution):
    """Validate the interface needed by visualization and animation."""
    missing = [
        name
        for name in REQUIRED_ROUNDTRIP_ATTRIBUTES
        if not hasattr(planner, name)
    ]
    if missing:
        raise AttributeError(
            "RoundtripPlanner is missing required attributes: "
            + ", ".join(missing)
        )

    if solution:
        unknown_nodes = [
            node for node in solution
            if node not in planner.graph
        ]
        if unknown_nodes:
            raise ValueError(
                "The solution contains nodes that are not in planner.graph."
            )

    if len(planner.planners) != len(planner.paths):
        raise ValueError(
            "planner.planners and planner.paths must have equal lengths."
        )

    return True


## 7. Experimente und Benchmarks

Beschreiben Sie Testumgebungen, Parameter, Metriken und Anzahl der Wiederholungen.

### 7.1 Benchmarks für den 2-DoF-Punktroboter
Der Punktroboter besitzt die Konfiguration $q=[x,y]$ und damit genau zwei Freiheitsgrade. Für die geforderten drei Fälle werden die Umgebungen **Trap**, **Bottleneck** und **Fat bottleneck** aus den Vorlesungs-Benchmarks wiederverwendet. Die Start- und Zielmengen werden für das Roundtrip-Problem auf mehrere Zielpunkte erweitert.

Zusätzlich werden die in `src.benchmarking.RoundtripTestSuite` definierten Benchmarks **Alternating Gates** und **Escape Chamber** unverändert übernommen. Damit wird der Punktroboter in insgesamt fünf unterschiedlichen Umgebungen getestet.


In [ ]:
import numpy as np

from notebooks.IPBenchmark import Benchmark
from notebooks.IPEnvironment import CollisionChecker
from src.benchmarking import RoundtripTestSuite as lecture_benchmarks


def copy_point_environment(lecture_benchmark):
    """Create an independent collision checker from a lecture benchmark."""
    checker = lecture_benchmark.collisionChecker
    return CollisionChecker(
        scene=dict(checker.scene),
        limits=checker.getEnvironmentLimits(),
    )


POINT_ROBOT_BENCHMARKS = [
    Benchmark(
        "PointRobot Trap",
        copy_point_environment(lecture_benchmarks.benchList[0]),
        [[10.0, 15.0]],
        [[10.0, 1.0], [2.0, 20.0], [20.0, 20.0]],
        "Roundtrip from inside the trap to three targets outside it.",
        2,
    ),
    Benchmark(
        "PointRobot Bottleneck",
        copy_point_environment(lecture_benchmarks.benchList[1]),
        [[4.0, 15.0]],
        [[18.0, 1.0], [4.0, 3.0], [18.0, 20.0], [8.0, 18.0]],
        "Roundtrip with targets on both sides of a narrow passage.",
        2,
    ),
    Benchmark(
        "PointRobot Fat Bottleneck",
        copy_point_environment(lecture_benchmarks.benchList[2]),
        [[4.0, 21.0]],
        [[18.0, 1.0], [4.0, 3.0], [18.0, 20.0], [12.0, 4.0]],
        "Roundtrip through an extended narrow passage.",
        3,
    ),
]

# Add the two new RoundtripTestSuite cases without redefining their geometry.
for benchmark_name in ("Alternating Gates", "Escape Chamber"):
    source_benchmark = next(
        benchmark
        for benchmark in lecture_benchmarks.benchList
        if benchmark.name == benchmark_name
    )

    POINT_ROBOT_BENCHMARKS.append(
        Benchmark(
            source_benchmark.name,
            copy_point_environment(source_benchmark),
            [list(configuration) for configuration in source_benchmark.startList],
            [list(configuration) for configuration in source_benchmark.goalList],
            source_benchmark.description,
            source_benchmark.level,
        )
    )


### 7.2 Benchmarks für den PlanarManipulator
Es werden drei Fälle definiert: ein einfacher 2-DoF-Fall, ein schwierigerer 2-DoF-Fall mit mehr Hindernissen und Zielen sowie ein 4-DoF-Fall. Ein fester Zufalls-Seed erzeugt reproduzierbare, kollisionsfreie und ausreichend voneinander entfernte Gelenkkonfigurationen.

In [ ]:
from shapely.geometry import LineString

from notebooks.IPEnvironmentKin import KinChainCollisionChecker
from notebooks.IPPlanarManipulator import PlanarRobot


LECTURE_PLANAR_SCENE = {
    "obs1": LineString([(-2.0, 0.0), (-0.8, 0.0)]).buffer(0.5),
    "obs2": LineString([(2.0, 0.0), (2.0, 1.0)]).buffer(0.2),
    "obs3": LineString([(-1.0, 2.0), (1.0, 2.0)]).buffer(0.1),
}


def angular_distance(first, second):
    """Return the wrapped Euclidean distance between joint configurations."""
    difference = np.asarray(first) - np.asarray(second)
    wrapped = (difference + np.pi) % (2.0 * np.pi) - np.pi
    return float(np.linalg.norm(wrapped))


def sample_free_configurations(
    environment,
    number_of_configurations,
    seed,
    minimum_distance=1.2,
    maximum_attempts=20_000,
):
    """Generate deterministic collision-free benchmark configurations."""
    generator = np.random.default_rng(seed)
    configurations = []

    for _ in range(maximum_attempts):
        candidate = generator.uniform(
            low=-np.pi,
            high=np.pi,
            size=environment.getDim(),
        ).tolist()

        if environment.pointInCollision(candidate):
            continue

        if any(
            angular_distance(candidate, existing) < minimum_distance
            for existing in configurations
        ):
            continue

        configurations.append(candidate)

        if len(configurations) == number_of_configurations:
            return configurations

    raise RuntimeError(
        f"Could not generate {number_of_configurations} free configurations."
    )


def create_planar_benchmark(
    name,
    degrees_of_freedom,
    scene,
    number_of_goals,
    seed,
    level,
    description,
):
    """Create a reproducible planar-manipulator roundtrip benchmark."""
    robot = PlanarRobot(n_joints=degrees_of_freedom)
    joint_limits = [[-np.pi, np.pi] for _ in range(degrees_of_freedom)]
    environment = KinChainCollisionChecker(
        robot,
        scene=dict(scene),
        limits=joint_limits,
        fk_resolution=0.15,
    )

    configurations = sample_free_configurations(
        environment,
        number_of_configurations=number_of_goals + 1,
        seed=seed,
    )

    return Benchmark(
        name,
        environment,
        [configurations[0]],
        configurations[1:],
        description,
        level,
    )


PLANAR_ROBOT_BENCHMARKS = [
    create_planar_benchmark(
        name="PlanarRobot 2-DoF Easy",
        degrees_of_freedom=2,
        scene={"obs2": LECTURE_PLANAR_SCENE["obs2"]},
        number_of_goals=3,
        seed=10,
        level=1,
        description="Two-joint planar robot with one workspace obstacle.",
    ),
    create_planar_benchmark(
        name="PlanarRobot 2-DoF Hard",
        degrees_of_freedom=2,
        scene=LECTURE_PLANAR_SCENE,
        number_of_goals=4,
        seed=21,
        level=3,
        description="Two-joint planar robot with the complete lecture scene.",
    ),
    create_planar_benchmark(
        name="PlanarRobot 4-DoF",
        degrees_of_freedom=4,
        scene=LECTURE_PLANAR_SCENE,
        number_of_goals=4,
        seed=42,
        level=3,
        description="Four-joint planar robot in a four-dimensional configuration space.",
    ),
]


benchmarks = POINT_ROBOT_BENCHMARKS + PLANAR_ROBOT_BENCHMARKS

### 7.3 Prüfung der Benchmarkdefinitionen

Vor der Planung werden Dimension und Kollisionsfreiheit aller Start- und Zielkonfigurationen geprüft. Dadurch werden ungültige Benchmarkdaten früh erkannt und nicht fälschlich als Fehler des Roundtrip-Planers gewertet.

In [ ]:
def validate_benchmark(benchmark):
    """Validate dimensions and collisions for all important configurations."""
    environment = benchmark.collisionChecker
    configurations = benchmark.startList + benchmark.goalList

    for configuration in configurations:
        if len(configuration) != environment.getDim():
            raise ValueError(
                f"{benchmark.name}: configuration {configuration} has the wrong dimension."
            )

        if environment.pointInCollision(configuration):
            raise ValueError(
                f"{benchmark.name}: configuration {configuration} is in collision."
            )

    return True


for benchmark in benchmarks:
    validate_benchmark(benchmark)
    robot_type = (
        "PlanarManipulator"
        if isinstance(benchmark.collisionChecker, KinChainCollisionChecker)
        else "PointRobot"
    )
    print(
        f"{benchmark.name:30s} | {robot_type:17s} | "
        f"DoF: {benchmark.collisionChecker.getDim()} | "
        f"Goals: {len(benchmark.goalList)}"
    )

### 7.4 Versuchsplan und reproduzierbare Einstellungen

Die Experimente werden über eine gemeinsame Konfiguration gesteuert. Während der Entwicklung wird nur ein Seed verwendet. Für die finale Auswertung sind zehn identische Seeds für alle Basisplaner vorgesehen. Dadurch werden die stochastischen Planer unter vergleichbaren Bedingungen untersucht. Die beiden Ein-Ziel-Benchmarks **Alternating Gates** und **Escape Chamber** bleiben zusätzliche Pfadplanungstests; die Mehrziel-Benchmarks untersuchen zusätzlich die Besuchsreihenfolge.


In [ ]:
# TASK2_EXPERIMENT_SETTINGS
import random
from time import perf_counter

import pandas as pd


BASE_PLANNERS_TO_COMPARE = tuple(BASE_PLANNER_CLASSES)
ORDER_METHODS_TO_COMPARE = ("brute_force", "greedy")

DEVELOPMENT_SEEDS = (17,)
FINAL_EXPERIMENT_SEEDS = (11, 17, 23, 31, 43, 59, 71, 83, 97, 109)

PRIMARY_POINT_BENCHMARKS = tuple(POINT_ROBOT_BENCHMARKS[:3])
SUPPLEMENTARY_POINT_BENCHMARKS = tuple(POINT_ROBOT_BENCHMARKS[3:])
PRIMARY_PLANAR_BENCHMARKS = tuple(PLANAR_ROBOT_BENCHMARKS)
ALL_EVALUATION_BENCHMARKS = tuple(benchmarks)


### 7.5 Einheitlicher Benchmark-Runner

Der Runner kapselt die einzige Abhängigkeit vom Roundtrip-Algorithmus. Er speichert erfolgreiche und fehlgeschlagene Versuche in derselben Struktur, misst die gesamte Planungszeit und bereitet alle geforderten Kennzahlen für die spätere Auswertung vor. Falls Dimas Implementierung andere Feldnamen verwendet, muss ausschließlich die Funktion `extract_experiment_metrics` angepasst werden.


In [ ]:
# TASK2_BENCHMARK_RUNNER
RESULT_COLUMNS = [
    "experiment_id",
    "benchmark",
    "robot_type",
    "dof",
    "number_of_goals",
    "difficulty",
    "base_planner",
    "order_method",
    "run",
    "seed",
    "success",
    "planning_time",
    "final_path_length",
    "final_path_points",
    "successful_subpaths",
    "failed_subpaths",
    "roadmap_nodes",
    "roadmap_edges",
    "collision_checks",
    "error",
]


def _read_value(source, names, default=None):
    """Read the first available value from a mapping or object."""
    if source is None:
        return default

    for name in names:
        if isinstance(source, dict) and name in source:
            return source[name]
        if hasattr(source, name):
            return getattr(source, name)

    return default


def normalize_plan_output(planner, plan_output):
    """Return a node path and metadata for common algorithm return formats."""
    if isinstance(plan_output, dict):
        solution = _read_value(
            plan_output,
            ("solution", "path", "final_node_path", "node_path"),
            [],
        )
        metadata = plan_output
    else:
        solution = plan_output
        metadata = {}

    if solution is None:
        solution = []

    return list(solution), metadata


def _path_positions(planner, solution, metadata):
    """Extract the final path as configurations."""
    positions = _read_value(
        metadata,
        ("final_path_positions", "path_positions", "config_path"),
    )
    if positions is None:
        positions = _read_value(
            planner,
            ("final_path_positions", "path_positions"),
        )

    if positions is None and planner is not None and hasattr(planner, "graph"):
        try:
            positions = [
                planner.graph.nodes[node]["pos"]
                for node in solution
            ]
        except (KeyError, TypeError):
            positions = []

    return [np.asarray(position, dtype=float) for position in (positions or [])]


def _path_length(positions):
    """Calculate Euclidean length when the planner supplies no length."""
    if len(positions) < 2:
        return 0.0
    return float(
        sum(
            np.linalg.norm(target - source)
            for source, target in zip(positions[:-1], positions[1:])
        )
    )


def _roadmap_size(planner):
    """Aggregate nodes and edges across all stored pairwise roadmaps."""
    pairwise_planners = _read_value(planner, ("planners",), []) or []
    graphs = [
        item.graph
        for item in pairwise_planners
        if hasattr(item, "graph")
    ]

    if not graphs and planner is not None and hasattr(planner, "graph"):
        graphs = [planner.graph]

    return (
        sum(graph.number_of_nodes() for graph in graphs),
        sum(graph.number_of_edges() for graph in graphs),
    )


def extract_experiment_metrics(
    planner,
    solution,
    metadata,
    benchmark,
    base_planner_name,
    order_method,
    run_index,
    seed,
    planning_time,
    experiment_id,
    error="",
):
    """Translate one algorithm run into the common evaluation schema."""
    planner_metrics = _read_value(planner, ("metrics",), {}) or {}
    sources = (metadata, planner_metrics, planner)

    def metric(names, default=None):
        for source in sources:
            value = _read_value(source, names)
            if value is not None:
                return value
        return default

    explicit_success = metric(("success",))
    success = bool(solution) if explicit_success is None else bool(explicit_success)
    if error:
        success = False

    positions = _path_positions(planner, solution, metadata)
    final_path_length = metric(("total_length", "final_path_length"))
    if final_path_length is None and success:
        final_path_length = _path_length(positions)

    pairwise_results = metric(("pairwise_results",), []) or []
    successful_subpaths = metric(("successful_subpaths", "successful_pairs"))
    failed_subpaths = metric(("failed_subpaths", "failed_pairs"))

    if successful_subpaths is None:
        if pairwise_results:
            successful_subpaths = sum(
                bool(_read_value(item, ("success",), False))
                for item in pairwise_results
            )
        else:
            successful_subpaths = len(
                _read_value(planner, ("paths",), []) or []
            )

    if failed_subpaths is None:
        pairwise_failures = metric(("pairwise_failures",), []) or []
        if pairwise_results:
            failed_subpaths = len(pairwise_results) - successful_subpaths
        else:
            failed_subpaths = len(pairwise_failures)

    roadmap_nodes = metric(("roadmap_nodes",))
    roadmap_edges = metric(("roadmap_edges",))
    if roadmap_nodes is None or roadmap_edges is None:
        computed_nodes, computed_edges = _roadmap_size(planner)
        roadmap_nodes = computed_nodes if roadmap_nodes is None else roadmap_nodes
        roadmap_edges = computed_edges if roadmap_edges is None else roadmap_edges

    collision_checks = metric(
        ("collision_checks", "number_of_collision_checks"),
        np.nan,
    )

    robot_type = (
        "PlanarManipulator"
        if isinstance(benchmark.collisionChecker, KinChainCollisionChecker)
        else "PointRobot"
    )

    return {
        "experiment_id": experiment_id,
        "benchmark": benchmark.name,
        "robot_type": robot_type,
        "dof": benchmark.collisionChecker.getDim(),
        "number_of_goals": len(benchmark.goalList),
        "difficulty": benchmark.level,
        "base_planner": base_planner_name,
        "order_method": order_method,
        "run": run_index,
        "seed": seed,
        "success": success,
        "planning_time": float(planning_time),
        "final_path_length": (
            float(final_path_length)
            if final_path_length is not None
            else np.nan
        ),
        "final_path_points": len(positions) if success else 0,
        "successful_subpaths": int(successful_subpaths or 0),
        "failed_subpaths": int(failed_subpaths or 0),
        "roadmap_nodes": int(roadmap_nodes or 0),
        "roadmap_edges": int(roadmap_edges or 0),
        "collision_checks": collision_checks,
        "error": error,
    }


def run_single_experiment(
    benchmark,
    base_planner_name,
    order_method,
    seed,
    run_index=0,
):
    """Run and store one complete RoundtripPlanner experiment."""
    experiment_id = (
        f"{benchmark.name}|{base_planner_name}|"
        f"{order_method}|run_{run_index:02d}|seed_{seed}"
    )

    random.seed(seed)
    np.random.seed(seed)
    planner = None
    solution = []
    metadata = {}
    error = ""
    config = build_roundtrip_config(
        base_planner_name,
        order_method,
        seed,
    )

    started = perf_counter()
    try:
        planner = create_roundtrip_planner(base_planner_name, benchmark)
        plan_output = planner.planPath(
            benchmark.startList,
            benchmark.goalList,
            config,
        )
        solution, metadata = normalize_plan_output(planner, plan_output)
        if solution:
            validate_roundtrip_interface(planner, solution)
    except Exception as exception:
        error = f"{type(exception).__name__}: {exception}"
    planning_time = perf_counter() - started

    metrics = extract_experiment_metrics(
        planner=planner,
        solution=solution,
        metadata=metadata,
        benchmark=benchmark,
        base_planner_name=base_planner_name,
        order_method=order_method,
        run_index=run_index,
        seed=seed,
        planning_time=planning_time,
        experiment_id=experiment_id,
        error=error,
    )

    record = {
        "planner": planner,
        "solution": solution,
        "benchmark": benchmark,
        "config": config,
        "metadata": metadata,
        "metrics": metrics,
    }
    roundtrip_results[experiment_id] = record
    return record


def run_benchmark_suite(
    selected_benchmarks,
    base_planner_names,
    order_methods,
    seeds,
):
    """Run the Cartesian product of the selected experiment settings."""
    for benchmark in selected_benchmarks:
        for base_planner_name in base_planner_names:
            for order_method in order_methods:
                for run_index, seed in enumerate(seeds):
                    record = run_single_experiment(
                        benchmark=benchmark,
                        base_planner_name=base_planner_name,
                        order_method=order_method,
                        seed=seed,
                        run_index=run_index,
                    )
                    status = "success" if record["metrics"]["success"] else "failed"
                    print(record["metrics"]["experiment_id"], status)

    return roundtrip_results


def find_roundtrip_result(
    benchmark_name,
    base_planner_name=None,
    order_method=None,
    successful_only=True,
):
    """Find one stored result suitable for visualization or animation."""
    for record in roundtrip_results.values():
        metrics = record["metrics"]
        if metrics["benchmark"] != benchmark_name:
            continue
        if base_planner_name and metrics["base_planner"] != base_planner_name:
            continue
        if order_method and metrics["order_method"] != order_method:
            continue
        if successful_only and not metrics["success"]:
            continue
        return record

    return None


In [ ]:
# TASK2_EXECUTION_CONTROL
roundtrip_results = {}

RUN_SMOKE_TEST = False
RUN_BENCHMARKS = False

BENCHMARKS_TO_RUN = ALL_EVALUATION_BENCHMARKS
BASE_PLANNERS_TO_RUN = BASE_PLANNERS_TO_COMPARE
ORDER_METHODS_TO_RUN = ORDER_METHODS_TO_COMPARE
SEEDS_TO_USE = DEVELOPMENT_SEEDS

if RUN_SMOKE_TEST:
    run_benchmark_suite(
        selected_benchmarks=(PRIMARY_POINT_BENCHMARKS[0],),
        base_planner_names=("BasicPRM",),
        order_methods=("brute_force",),
        seeds=DEVELOPMENT_SEEDS,
    )
elif RUN_BENCHMARKS:
    run_benchmark_suite(
        selected_benchmarks=BENCHMARKS_TO_RUN,
        base_planner_names=BASE_PLANNERS_TO_RUN,
        order_methods=ORDER_METHODS_TO_RUN,
        seeds=SEEDS_TO_USE,
    )
else:
    print(
        "Benchmark execution is disabled. Configure Dima's algorithm and "
        "set RUN_SMOKE_TEST or RUN_BENCHMARKS to True."
    )


## 8. Visualisierungen und Animationen

Zeigen Sie Suchraum, Roadmap/Baum, Pfad, Kollisionen, Statistiken oder Animationen.

In [ ]:
# Visualization and animation helpers are defined below.

### 8.1 Visualisierung der Roundtrip-Ergebnisse
Die vorhandenen Visualisierungsfunktionen aus `IPVISBasicPRM.py`, `IPVISLazyPRM.py` und `IPVISVisibilityPRM.py` werden direkt wiederverwendet. Für den zusammengesetzten Roundtrip wird `basicPRMVisualize` genutzt, da diese Funktion nur den Graphen, die Kollisionsprüfung und den Lösungspfad benötigt.

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

from notebooks.IPVISBasicPRM import basicPRMVisualize
from notebooks.IPVISLazyPRM import lazyPRMVisualize
from notebooks.IPVISVisibilityPRM import visibilityPRMVisualize


PLANNER_VISUALIZERS = {
    "BasicPRM": basicPRMVisualize,
    "LazyPRM": lazyPRMVisualize,
    "VisPRM": visibilityPRMVisualize,
    "VisibilityPRM": visibilityPRMVisualize,
}


def _configuration_labels(benchmark):
    """Map start and goal configurations to readable labels."""
    labels = {tuple(np.asarray(benchmark.startList[0], dtype=float)): "S"}
    for index, goal in enumerate(benchmark.goalList, start=1):
        labels[tuple(np.asarray(goal, dtype=float))] = f"G{index}"
    return labels


def _visit_order(planner, solution, benchmark):
    """Extract the visit order from the final graph path."""
    labels = _configuration_labels(benchmark)
    order = []

    for node in solution:
        position = tuple(np.asarray(planner.graph.nodes[node]["pos"], dtype=float))
        label = labels.get(position)
        if label is not None and (not order or order[-1] != label):
            order.append(label)

    return order


def get_base_visualizer(roundtrip_planner, fallback_name=None):
    """Select the lecture visualizer matching the configured base planner."""
    planner_class = getattr(roundtrip_planner, "plannerClass", None)
    planner_name = getattr(planner_class, "__name__", fallback_name)
    if planner_name not in PLANNER_VISUALIZERS:
        raise KeyError(f"No visualizer registered for {planner_name!r}.")
    return PLANNER_VISUALIZERS[planner_name]


def visualize_roundtrip(planner, solution, benchmark):
    """Show start, goals, selected order, and final path for a 2-DoF case."""
    positions = nx.get_node_attributes(planner.graph, "pos")
    if any(len(np.asarray(position)) != 2 for position in positions.values()):
        raise ValueError(
            "Static roundtrip visualization requires a 2-DoF configuration space."
        )

    fig, ax = plt.subplots(figsize=(10, 8))
    basicPRMVisualize(planner, solution, ax=ax, nodeSize=45)

    labels_by_configuration = _configuration_labels(benchmark)
    important_nodes = {}
    for node, position in positions.items():
        label = labels_by_configuration.get(tuple(np.asarray(position, dtype=float)))
        if label is not None:
            important_nodes[node] = label

    start_nodes = [node for node, label in important_nodes.items() if label == "S"]
    goal_nodes = [node for node, label in important_nodes.items() if label != "S"]

    nx.draw_networkx_nodes(
        planner.graph,
        positions,
        nodelist=start_nodes,
        node_color="#00aa44",
        node_size=240,
        node_shape="*",
        ax=ax,
    )
    nx.draw_networkx_nodes(
        planner.graph,
        positions,
        nodelist=goal_nodes,
        node_color="#e53935",
        node_size=170,
        ax=ax,
    )
    nx.draw_networkx_labels(planner.graph, positions, labels=important_nodes, ax=ax)

    order = _visit_order(planner, solution, benchmark)
    ax.set_title(
        f"{benchmark.name}\nVisit order: " + " -> ".join(order),
        fontweight="bold",
    )
    ax.set_aspect("equal", adjustable="box")
    return fig, ax


def visualize_pairwise_paths(roundtrip_planner, visualizer):
    """Show every pairwise path generated by the selected base planner."""
    number_of_paths = len(roundtrip_planner.paths)
    if number_of_paths == 0:
        raise ValueError("No pairwise paths are available for visualization.")

    columns = min(3, number_of_paths)
    rows = math.ceil(number_of_paths / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(6 * columns, 5 * rows))
    axes = np.atleast_1d(axes).ravel()

    for index, (planner, path) in enumerate(
        zip(roundtrip_planner.planners, roundtrip_planner.paths)
    ):
        visualizer(planner, path, ax=axes[index], nodeSize=35)
        axes[index].set_title(f"Pairwise path {index + 1}")
        axes[index].set_aspect("equal", adjustable="box")

    for axis in axes[number_of_paths:]:
        axis.set_visible(False)

    fig.tight_layout()
    return fig, axes


### 8.2 Animation für Punktroboter und PlanarManipulator

Für den Punktroboter wird der finale Roundtrip direkt im Arbeitsraum animiert. Bei einem 2-DoF-PlanarManipulator zeigt die vorhandene Vorlesungsfunktion `animateSolution` den Arbeitsraum und den zweidimensionalen Konfigurationsraum nebeneinander. Bei vier oder mehr Freiheitsgraden wird ausschließlich die Roboterbewegung im Arbeitsraum dargestellt.


In [ ]:
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

from notebooks.IPEnvironmentKin import KinChainCollisionChecker, animateSolution


def _solution_positions(planner, solution):
    """Return the configurations represented by a graph-node path."""
    return np.asarray(
        [planner.graph.nodes[node]["pos"] for node in solution],
        dtype=float,
    )


def _interpolate_configurations(positions, frames_per_segment=20):
    """Create uniformly interpolated configurations for smooth animation."""
    interpolated = [positions[0]]
    for source, target in zip(positions[:-1], positions[1:]):
        segment = np.linspace(source, target, frames_per_segment + 1)
        interpolated.extend(segment[1:])
    return np.asarray(interpolated)


def animate_point_roundtrip(
    roundtrip_planner,
    solution,
    benchmark,
    frames_per_segment=20,
):
    """Animate a complete PointRobot roundtrip in the workspace."""
    environment = benchmark.collisionChecker
    if isinstance(environment, KinChainCollisionChecker):
        raise TypeError("The benchmark must use the PointRobot collision checker.")

    positions = _solution_positions(roundtrip_planner, solution)
    if positions.ndim != 2 or positions.shape[1] != 2:
        raise ValueError("PointRobot animation requires 2D configurations.")

    frames = _interpolate_configurations(positions, frames_per_segment)
    limits = environment.getEnvironmentLimits()
    fig, ax = plt.subplots(figsize=(8, 7))

    def draw_frame(frame_index):
        ax.clear()
        environment.drawObstacles(ax)
        ax.set_xlim(limits[0])
        ax.set_ylim(limits[1])
        ax.set_aspect("equal", adjustable="box")
        ax.plot(positions[:, 0], positions[:, 1], "--", color="#777777")
        ax.plot(
            frames[: frame_index + 1, 0],
            frames[: frame_index + 1, 1],
            color="#1565c0",
            linewidth=2.5,
        )
        ax.scatter(
            benchmark.startList[0][0],
            benchmark.startList[0][1],
            marker="*",
            s=220,
            color="#00aa44",
            label="Start",
        )
        goals = np.asarray(benchmark.goalList, dtype=float)
        ax.scatter(goals[:, 0], goals[:, 1], color="#e53935", label="Goals")
        ax.scatter(
            frames[frame_index, 0],
            frames[frame_index, 1],
            s=120,
            color="#ff9800",
            label="Robot",
            zorder=10,
        )
        ax.set_title(benchmark.name)
        ax.legend(loc="upper right")

    animation = FuncAnimation(fig, draw_frame, frames=len(frames))
    html = HTML(animation.to_jshtml())
    plt.close(fig)
    return html


def animate_planar_roundtrip(
    roundtrip_planner,
    solution,
    benchmark,
    workspace_limits,
):
    """Animate a complete planar-manipulator roundtrip in the workspace."""
    environment = benchmark.collisionChecker
    if not isinstance(environment, KinChainCollisionChecker):
        raise TypeError("The benchmark must use KinChainCollisionChecker.")

    expected_dimension = environment.getDim()
    for position in _solution_positions(roundtrip_planner, solution):
        if len(position) != expected_dimension:
            raise ValueError(
                "Planner solution and PlanarRobot have different dimensions."
            )

    return animateSolution(
        roundtrip_planner,
        environment,
        solution,
        basicPRMVisualize,
        workSpaceLimits=workspace_limits,
    )


### 8.3 Anwendung auf die Versuchsergebnisse

Ein gespeicherter Versuch wird anhand von Benchmark, Basisplaner und Reihenfolgeverfahren ausgewählt. Solange der Algorithmus noch nicht eingebunden oder kein erfolgreicher Lauf vorhanden ist, wird die Zelle ohne Fehler übersprungen. Statische Darstellungen und Animationen können unabhängig voneinander aktiviert werden.


In [ ]:
VISUALIZATION_SELECTION = {
    "benchmark_name": "PlanarRobot 2-DoF Easy",
    "base_planner_name": "VisibilityPRM",
    "order_method": "brute_force",
}

CREATE_STATIC_VISUALIZATIONS = True
CREATE_ANIMATION = False

selected_result = find_roundtrip_result(**VISUALIZATION_SELECTION)

if selected_result is None:
    print(
        "No matching successful result is available. Run the selected "
        "benchmark configuration first."
    )
else:
    roundtrip_planner = selected_result["planner"]
    solution = selected_result["solution"]
    benchmark = selected_result["benchmark"]
    base_planner_name = selected_result["metrics"]["base_planner"]

    if CREATE_STATIC_VISUALIZATIONS:
        base_visualizer = get_base_visualizer(
            roundtrip_planner,
            fallback_name=base_planner_name,
        )
        if benchmark.collisionChecker.getDim() == 2:
            visualize_roundtrip(roundtrip_planner, solution, benchmark)
        visualize_pairwise_paths(roundtrip_planner, base_visualizer)

    if CREATE_ANIMATION:
        if isinstance(benchmark.collisionChecker, KinChainCollisionChecker):
            animate_planar_roundtrip(
                roundtrip_planner,
                solution,
                benchmark,
                workspace_limits=[[-5, 5], [-5, 5]],
            )
        else:
            display(
                animate_point_roundtrip(
                    roundtrip_planner,
                    solution,
                    benchmark,
                )
            )


## 9. Ergebnisse

Alle gespeicherten Versuche werden in eine gemeinsame Tabelle überführt. Dieselben Rohdaten bilden sowohl die Evaluation der Benchmarkfälle als auch den Vergleich der Basisplaner. Pfadlänge und Pfadpunkte werden nur für erfolgreiche Läufe interpretiert; die Planungszeit und Erfolgsquote berücksichtigen auch fehlgeschlagene Läufe.


In [ ]:
# TASK2_RESULT_ANALYSIS
from pathlib import Path

from IPython.display import display


def results_dataframe(results):
    """Convert stored experiment records into a stable DataFrame schema."""
    rows = [record["metrics"] for record in results.values()]
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)


def summarize_results(dataframe):
    """Aggregate repeated runs for the report tables."""
    if dataframe.empty:
        return pd.DataFrame()

    group_columns = [
        "robot_type",
        "benchmark",
        "base_planner",
        "order_method",
    ]
    return (
        dataframe.groupby(group_columns, dropna=False)
        .agg(
            runs=("success", "size"),
            success_rate=("success", "mean"),
            planning_time_mean=("planning_time", "mean"),
            planning_time_std=("planning_time", "std"),
            final_path_length_mean=("final_path_length", "mean"),
            final_path_points_mean=("final_path_points", "mean"),
            successful_subpaths_mean=("successful_subpaths", "mean"),
            failed_subpaths_mean=("failed_subpaths", "mean"),
            roadmap_nodes_mean=("roadmap_nodes", "mean"),
            roadmap_edges_mean=("roadmap_edges", "mean"),
            collision_checks_mean=("collision_checks", "mean"),
        )
        .reset_index()
    )


def plot_success_rate(dataframe):
    """Compare success rates by base planner and visit-order method."""
    rates = (
        dataframe.groupby(["base_planner", "order_method"])["success"]
        .mean()
        .mul(100.0)
        .unstack("order_method")
    )
    ax = rates.plot(kind="bar", figsize=(9, 5), ylim=(0, 100))
    ax.set_ylabel("Success rate [%]")
    ax.set_title("Roundtrip success rate")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    return ax


def plot_planner_metric(dataframe, metric, ylabel, successful_only=False):
    """Compare one measured metric across base planners."""
    selected = dataframe
    if successful_only:
        selected = selected[selected["success"]]
    selected = selected.dropna(subset=[metric])
    if selected.empty:
        raise ValueError(f"No values are available for {metric!r}.")

    values = (
        selected.groupby(["base_planner", "order_method"])[metric]
        .mean()
        .unstack("order_method")
    )
    ax = values.plot(kind="bar", figsize=(9, 5))
    ax.set_ylabel(ylabel)
    ax.set_title(metric.replace("_", " ").title())
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    return ax


def save_results_csv(dataframe, path="results/roundtrip_results.csv"):
    """Save raw experiment metrics without serializing planner objects."""
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(output_path, index=False)
    return output_path


results_df = results_dataframe(roundtrip_results)
summary_df = summarize_results(results_df)

if results_df.empty:
    print("No benchmark results are available yet.")
else:
    display(results_df.head())
    display(summary_df)
    plot_success_rate(results_df)
    plot_planner_metric(
        results_df,
        metric="planning_time",
        ylabel="Planning time [s]",
    )
    plot_planner_metric(
        results_df,
        metric="final_path_length",
        ylabel="Final roundtrip length",
        successful_only=True,
    )
    plot_planner_metric(
        results_df,
        metric="collision_checks",
        ylabel="Collision checks",
    )


## 10. Diskussion

Diskutieren Sie, was funktioniert hat, wo Grenzen liegen, welche Parameter wichtig sind und wie belastbar die Ergebnisse sind.

## 11. Fazit

Fassen Sie die wichtigsten Erkenntnisse knapp zusammen.

## 12. Verwendung von KI-Werkzeugen

Dokumentieren Sie, wofür KI verwendet wurde, welche Vorschläge übernommen oder verworfen wurden und wie die Korrektheit geprüft wurde.

## 13. Präsentationsnotizen

Notieren Sie die Kernaussagen für die Präsentation: Problem, Ansatz, wichtigste Visualisierung, wichtigste Ergebnisse und wichtigste Erkenntnis.